# 01 — Data Collection
**Proyek:** Klasifikasi Tingkat Risiko Stroke Berdasarkan Data Klinis dan Gaya Hidup Menggunakan Algoritma TabNet dengan Interpretasi Attention Mechanism

**Tahap:** Data Collection


## 1. Import Library

In [1]:
import hashlib
import json
import os
from datetime import datetime
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


## 2. Konfigurasi Path Proyek

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
METADATA_DIR = PROJECT_ROOT / "metadata"
DATA_RAW.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

DATASET_FILE = DATA_RAW / "healthcare-dataset-stroke-data.csv"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_FILE:", DATASET_FILE)
print("Exists       :", DATASET_FILE.exists())


PROJECT_ROOT: /Users/macbookpro/Documents/Kuliah/Semester 6/Proyek Data Mining/Riset Data Mining
DATASET_FILE: /Users/macbookpro/Documents/Kuliah/Semester 6/Proyek Data Mining/Riset Data Mining/data/raw/healthcare-dataset-stroke-data.csv
Exists       : True


## 3. Informasi Sumber Data

Dataset diambil dari Kaggle dengan detail sebagai berikut:

| Atribut | Keterangan |
|---|---|
| Nama | Stroke Prediction Dataset |
| Publisher | fedesoriano |
| URL | https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset |
| Lisensi | Data files © Original Authors |
| Ukuran | ± 316 KB (CSV) |
| Tahun Rilis | 2021 |
| Jumlah Record | 5.110 pasien |
| Jumlah Fitur | 11 fitur + 1 target |

Dataset memuat atribut **klinis** (usia, hipertensi, penyakit jantung, level glukosa, BMI) dan **gaya hidup** (status perkawinan, tipe pekerjaan, tempat tinggal, status merokok) yang relevan untuk memprediksi kejadian stroke.


## 4. Load Dataset

In [3]:
df = pd.read_csv(DATASET_FILE)
print(f"Shape       : {df.shape}")
print(f"Kolom       : {list(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
df.head()


Shape       : (5110, 12)
Kolom       : ['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']
Memory usage: 1657.67 KB


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 5. Validasi Integritas File (SHA-256)

In [4]:
def file_sha256(path: Path, chunk: int = 1 << 16) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

file_hash = file_sha256(DATASET_FILE)
file_size_bytes = DATASET_FILE.stat().st_size

print(f"SHA-256 : {file_hash}")
print(f"Size    : {file_size_bytes:,} bytes  ({file_size_bytes/1024:.2f} KB)")


SHA-256 : 644d473b05d2797006bd94865e4f8bb057f0c721617911613c82c8fcfc707420
Size    : 316,971 bytes  (309.54 KB)


## 6. Schema & Tipe Data Awal

In [5]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "null": df.isna().sum(),
    "n_unique": df.nunique(),
    "example": df.iloc[0].astype(str).values,
})
schema


,dtype,non_null,null,n_unique,example
id,int64,5110,0,5110,9046
gender,object,5110,0,3,Male
age,float64,5110,0,104,67.0
hypertension,int64,5110,0,2,0
heart_disease,int64,5110,0,2,1
ever_married,object,5110,0,2,Yes
work_type,object,5110,0,5,Private
Residence_type,object,5110,0,2,Urban
avg_glucose_level,float64,5110,0,3979,228.69
bmi,float64,4909,201,418,36.6


**Catatan penting:** Kolom `bmi` memiliki nilai `N/A` (string) pada file mentah. Ini akan terbaca sebagai `NaN` oleh pandas karena `read_csv` secara default mengenali string `"N/A"` sebagai missing. Begitu juga kolom `smoking_status` memiliki kategori `"Unknown"` yang secara semantik adalah *missing* namun tidak kosong.

## 7. Ringkasan Target

In [6]:
target_counts = df["stroke"].value_counts().rename({0: "Tidak Stroke", 1: "Stroke"})
target_ratio = df["stroke"].value_counts(normalize=True).mul(100).round(2)
summary = pd.DataFrame({"count": target_counts, "percent": target_ratio.values})
summary


,count,percent
stroke,,
Tidak Stroke,4861,95.13
Stroke,249,4.87


Dataset sangat **tidak seimbang** (imbalanced) — proporsi kelas positif (stroke) hanya ± 4.87%. Hal ini harus ditangani di tahap preprocessing (SMOTE / class weighting).

## 8. Simpan Metadata Collection

In [7]:
metadata = {
    "dataset_name": "healthcare-dataset-stroke-data",
    "source": {
        "platform": "Kaggle",
        "publisher": "fedesoriano",
        "url": "https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset",
        "license": "Original authors",
    },
    "collection": {
        "collected_at": datetime.utcnow().isoformat() + "Z",
        "collector": os.environ.get("USER", "unknown"),
        "local_path": str(DATASET_FILE.relative_to(PROJECT_ROOT)),
    },
    "integrity": {
        "sha256": file_hash,
        "size_bytes": file_size_bytes,
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
    },
    "target": {
        "name": "stroke",
        "type": "binary",
        "positive_rate": float(df["stroke"].mean()),
        "class_counts": df["stroke"].value_counts().to_dict(),
    },
    "features": [
        {"name": c, "dtype": str(df[c].dtype), "n_unique": int(df[c].nunique())}
        for c in df.columns
    ],
}

metadata_path = METADATA_DIR / "01_data_collection.json"
with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"Metadata collection tersimpan → {metadata_path}")


Metadata collection tersimpan → /Users/macbookpro/Documents/Kuliah/Semester 6/Proyek Data Mining/Riset Data Mining/metadata/01_data_collection.json


/var/folders/fr/361w11bj6cj7q3c4n1y75k4w0000gn/T/ipykernel_98646/2296484882.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "collected_at": datetime.utcnow().isoformat() + "Z",
